In [11]:
import truststore
truststore.inject_into_ssl()

import requests
import json
import os
import csv
import time
from datetime import datetime, timedelta, timezone


API_KEY = '5b4c8c74-748f-47da-89c6-21ec21da6bd5'

# Un día reciente, para permanecer dentro de los 30 días gratuitos
ayer = (datetime.now(timezone.utc) - timedelta(days=1)).date().isoformat()

url = "https://eventregistry.org/api/v1/article/getArticles"

### Codigo 1: Test para ver output generales

In [ ]:
params = {
    "apiKey": API_KEY,
    "keyword": [
        "gold prices",
        "gold futures",
        "spot gold",
        "gold bullion",
        "XAU/USD",
        "gold market"
    ],
    "keywordOper": "or",
    "keywordLoc": "title",
    "lang": "eng",
    "dateStart": ayer,
    "dateEnd": ayer,
    "articlesCount": 20,
    "articlesPage": 1,
    "articlesSortBy": "date",
    "articleBodyLen": -1,
    "includeArticleBody": True,
    "includeArticleEventUri": True,
    "isDuplicateFilter": "keepAll",
    "resultType": "articles"
}

In [9]:
r = requests.get(url, params=params, timeout=45)
print("HTTP:", r.status_code)
r.raise_for_status()

data = r.json()

# Guardar respuesta íntegra para inspección
with open("../data/prueba_newsapi.json", "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=2)

bloque = data.get("articles", {})
articulos = bloque.get("results", [])

HTTP: 200


In [10]:
print("Total encontrados:", bloque.get("totalResults"))
print("Páginas:", bloque.get("pages"))
print("Artículos recibidos:", len(articulos))
print("Costo estimado: 1 token")

for i, a in enumerate(articulos, 1):
    print(f"\n{'='*60}")
    print("ARTÍCULO", i)
    print("URI:", a.get("uri"))
    print("Fecha:", a.get("dateTime"))
    print("Medio:", (a.get("source") or {}).get("title"))
    print("Evento:", a.get("eventUri"))
    print("Duplicado:", a.get("isDuplicate"))
    print("Título:", a.get("title"))
    print("Texto:", (a.get("body") or "")[:1500])

print("\nJSON guardado en prueba_newsapi.json")


Total encontrados: 52
Páginas: 3
Artículos recibidos: 20
Costo estimado: 1 token

ARTÍCULO 1
URI: 9497092466
Fecha: 2026-10-07T21:19:50Z
Medio: EconoTimes
Evento: eng-12068716
Duplicado: False
Título: Gold Prices Fall as Fed Signals Another Rate Hike
Texto: Gold Prices Fall as Fed Signals Another Rate Hike. Source: Image by Robert Owen-Wahl from Pixabay

Gold prices trimmed some losses on Wednesday after Federal Reserve minutes showed that most U.S. policymakers expect another interest rate hike before the end of the year, reinforcing concerns over higher borrowing costs.

Spot gold fell 1.8% to $4,137.43, while gold futures declined by the same percentage to $4,113.76 as of 14:34 ET (18:34 GMT). The precious metal remained under pressure as the U.S. dollar strengthened following the Fed minutes.

Minutes from the Federal Reserve's September meeting showed that most policymakers anticipate another rate increase this year. At that meeting, the Federal Open Market Committee unanimously r

### Código 2: Testeemos familias de peticiones

In [12]:

FAMILIAS = {
    "oro": [
    "gold prices",
    "gold futures",
    "spot gold",
    "gold bullion",
    "XAU/USD",
    "gold market"
],
    "politica_monetaria": [
        "Federal Reserve", "FOMC", "interest rates", "rate cuts"
    ],
    "inflacion": ["inflation", "CPI", "PCE"],
    "dolar_bonos": ["US dollar", "Treasury yields", "bond yields"],
    "geopolitica": [
        "geopolitical risk", "sanctions", "armed conflict"
    ]
}

# Período inicial de 7 días, dentro de la ventana gratuita
hoy = datetime.now(timezone.utc).date()
INICIO = (hoy - timedelta(days=7)).isoformat()
FIN = (hoy - timedelta(days=1)).isoformat()

MAX_PAGINAS = 2       # 2 x 100 = máximo 200 por familia
ARTICULOS_PAGINA = 100
PAUSA_SEGUNDOS = 1.0

registros = {}
peticiones = 0

for familia, keywords in FAMILIAS.items():
    print(f"\n{'='*65}", flush=True)
    print(f"FAMILIA: {familia}", flush=True)
    print(f"Keywords: {keywords}", flush=True)

    total_familia = 0

    for pagina in range(1, MAX_PAGINAS + 1):
        params = {
            "apiKey": API_KEY,
            "keyword": keywords,
            "keywordOper": "or",
            "keywordLoc": "title",
            "lang": "eng",
            "dateStart": INICIO,
            "dateEnd": FIN,
            "articlesCount": ARTICULOS_PAGINA,
            "articlesPage": pagina,
            "articlesSortBy": "date",
            "articleBodyLen": -1,
            "includeArticleBody": True,
            "includeArticleEventUri": True,
            "isDuplicateFilter": "keepAll",
            "resultType": "articles"
        }

        # Peticiones secuenciales, sin paralelismo
        for intento in range(3):
            try:
                r = requests.get(url, params=params, timeout=60)

                if r.status_code == 429:
                    espera = 5 * (2 ** intento)
                    print(f"  HTTP 429. Esperando {espera}s...")
                    time.sleep(espera)
                    continue

                r.raise_for_status()
                data = r.json()
                break

            except requests.RequestException as e:
                if intento == 2:
                    raise
                print(f"  Error: {e}. Reintentando...")
                time.sleep(5 * (2 ** intento))
        else:
            print("  No se pudo recuperar la página.")
            continue

        peticiones += 1
        bloque = data.get("articles", {})
        articulos = bloque.get("results", [])

        for a in articulos:
            uri = a.get("uri")
            if not uri:
                continue

            if uri not in registros:
                registros[uri] = {
                    "uri": uri,
                    "dateTime": a.get("dateTime"),
                    "source": (a.get("source") or {}).get("title"),
                    "eventUri": a.get("eventUri"),
                    "isDuplicate": a.get("isDuplicate"),
                    "title": a.get("title"),
                    "body": a.get("body"),
                    "url": a.get("url"),
                    "familias": []
                }

            if familia not in registros[uri]["familias"]:
                registros[uri]["familias"].append(familia)

        total_familia += len(articulos)

        print(
            f"  Página {pagina} | "
            f"Recibidos: {len(articulos)} | "
            f"Total API: {bloque.get('totalResults')} | "
            f"Únicos acumulados: {len(registros)} | "
            f"Tokens estimados: {peticiones}",
            flush=True
        )

        if not articulos or len(articulos) < ARTICULOS_PAGINA:
            break

        time.sleep(PAUSA_SEGUNDOS)

    print(f"Finalizada {familia}: {total_familia} artículos recibidos")

# Guardar JSONL: útil para procesamiento con LLM
with open("noticias_familias.jsonl", "w", encoding="utf-8") as f:
    for a in registros.values():
        f.write(json.dumps(a, ensure_ascii=False) + "\n")

# Guardar CSV: útil para pandas, Excel o R
campos = [
    "uri", "dateTime", "source", "eventUri",
    "isDuplicate", "title", "body", "url", "familias"
]

with open("../data/noticias_familias.csv", "w",
          encoding="utf-8-sig", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=campos)
    writer.writeheader()

    for a in registros.values():
        fila = a.copy()
        fila["familias"] = "|".join(fila["familias"])
        writer.writerow(fila)

print("\n" + "=" * 65)
print("EXTRACCIÓN FINALIZADA")
print("Solicitudes exitosas:", peticiones)
print("Tokens estimados:", peticiones)
print("Artículos únicos:", len(registros))
print("Con texto:", sum(bool(a["body"]) for a in registros.values()))
print("Con evento:", sum(bool(a["eventUri"]) for a in registros.values()))
print("Archivos: noticias_familias.csv / noticias_familias.jsonl")



FAMILIA: oro
Keywords: ['gold prices', 'gold futures', 'spot gold', 'gold bullion', 'XAU/USD', 'gold market']
  Página 1 | Recibidos: 100 | Total API: 248 | Únicos acumulados: 100 | Tokens estimados: 1
  Página 2 | Recibidos: 100 | Total API: 248 | Únicos acumulados: 200 | Tokens estimados: 2
Finalizada oro: 200 artículos recibidos

FAMILIA: politica_monetaria
Keywords: ['Federal Reserve', 'FOMC', 'interest rates', 'rate cuts']
  Página 1 | Recibidos: 100 | Total API: 499 | Únicos acumulados: 300 | Tokens estimados: 3
  Página 2 | Recibidos: 100 | Total API: 499 | Únicos acumulados: 400 | Tokens estimados: 4
Finalizada politica_monetaria: 200 artículos recibidos

FAMILIA: inflacion
Keywords: ['inflation', 'CPI', 'PCE']
  Página 1 | Recibidos: 100 | Total API: 2539 | Únicos acumulados: 499 | Tokens estimados: 5
  Página 2 | Recibidos: 100 | Total API: 2539 | Únicos acumulados: 598 | Tokens estimados: 6
Finalizada inflacion: 200 artículos recibidos

FAMILIA: dolar_bonos
Keywords: ['US d